### Zadanie 
#### Zastosowanie algorytmu LIME do wyjaśnienia, dlaczego model klasyfikacyjny przewiduje, że dany pasażer Titanica przeżył katastrofę lub nie. Analiza ma pomóc zidentyfikować cechy (np. wiek, płeć, klasa podróży) najbardziej wpływające na decyzje modelu.

### Klasyfikacja za pomocą klasycznego modelu uczenia maszynowego  
#### 1. Importowanie odpowiednich bibliotek.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, ElasticNet, Ridge
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, classification_report, roc_curve, auc, root_mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn import datasets, svm


2. Przygotowanie danych.

In [23]:
df = pd.read_csv('titanic_train.csv')
df.head()
print(df.describe())

       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071   14.526497    1.102743   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     223.500000    0.000000    2.000000   20.125000    0.000000   
50%     446.000000    0.000000    3.000000   28.000000    0.000000   
75%     668.500000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            Parch        Fare  
count  891.000000  891.000000  
mean     0.381594   32.204208  
std      0.806057   49.693429  
min      0.000000    0.000000  
25%      0.000000    7.910400  
50%      0.000000   14.454200  
75%      0.000000   31.000000  
max      6.000000  512.329200  


In [24]:
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [25]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df.drop('Cabin', axis=1, inplace=True)
df.dropna(subset=['Embarked'], inplace = True)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
df.drop(columns=['Name', 'Ticket'], inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_16836\744521504.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)


In [26]:
df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,1,0,3,0,22.0,1,0,7.2500,False,True
1,2,1,1,1,38.0,1,0,71.2833,False,False
2,3,1,3,1,26.0,0,0,7.9250,False,True
3,4,1,1,1,35.0,1,0,53.1000,False,True
4,5,0,3,0,35.0,0,0,8.0500,False,True


3. Podział zbioru na macierz cech (X) oraz wektor wartości docelowych (y).

In [27]:
X = df.drop(columns=['Survived'], axis=1)
y = df['Survived']

4. Podziału zbioru na zbiór treningowy i testowy. Na podstawie uczenia zbioru treningowego, należy dokonać predykcji na temat przeżycia/śmierci pasażerów Titanica ze zbioru testowego. Zbiór testowy ma składać się z niemniej niż 200 przypadków. Zarodek liczb losowych ma być różny od 101.

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=200 / len(df), random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((689, 9), (200, 9), (689,), (200,))

##### 5. Zbudowanie modelu z wyłączeniem regresji logistycznej.
##### 6. Wytrenowanie modelu.
##### 7. Wypróbowanie modelu na zbiorze testowym. 
##### 8. Ewaluacja modelu na podstawie raportu z klasyfikacji i macierzy błędów.

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print("Raport z klasyfikacji:\n", class_report)

print("\nMacierz błędów:")
print(conf_matrix)


Raport z klasyfikacji:
               precision    recall  f1-score   support

           0       0.82      0.86      0.84       125
           1       0.74      0.69      0.72        75

    accuracy                           0.80       200
   macro avg       0.78      0.77      0.78       200
weighted avg       0.79      0.80      0.79       200


Macierz błędów:
[[107  18]
 [ 23  52]]


## Wyjaśnienie klasyfikacji za pomocą algorytmu LIME 
#### 1. Importowanie odpowiednich bibliotek.

In [30]:
import lime
from lime.lime_tabular import LimeTabularExplainer

2. Wykorzystanie klasy LimeTabularExplainer do stworzenia obiektu explainer, trenowanego na zbiorze uczącym. 

In [31]:
explainer = LimeTabularExplainer(
        X_train.values,
        mode="classification",
        feature_names=X.columns,
        class_names=["Did not Survive", "Survived"],
        discretize_continuous=True,
        random_state=42
    )

explainer

3. Wyjaśnienie wyników klasyfikacji dla czterech różnych przypadków: TN, FP, FN i TP uwzględniając prawdopodobieństwo oraz cechy mające największy wpływ na predykcję. Dla każdego przypadku należy podać:
- Etykietę rzeczywistą,
- Etykietę przewidzianą
- Indeks obserwacji.

In [32]:
# Znajdowanie przypadków TN, FP, FN, TP
y_test = y_test.reset_index(drop=True)
y_pred = pd.Series(y_pred).reset_index(drop=True)

true_negative_indices = (y_test == 0) & (y_pred == 0)
false_positive_indices = (y_test == 0) & (y_pred == 1)
false_negative_indices = (y_test == 1) & (y_pred == 0)
true_positive_indices = (y_test == 1) & (y_pred == 1)

tn_index = true_negative_indices[true_negative_indices].index[0]
fp_index = false_positive_indices[false_positive_indices].index[0]
fn_index = false_negative_indices[false_negative_indices].index[0]
tp_index = true_positive_indices[true_positive_indices].index[0]

lime_results = {}

for case, index in zip(["TN", "FP", "FN", "TP"], [tn_index, fp_index, fn_index, tp_index]):
    explanation = explainer.explain_instance(
        X_test.iloc[index].values,
        model.predict_proba,
        num_features=5
    )
    lime_results[case] = {
        "index": index,
        "actual": y_test[index],
        "predicted": y_pred[index],
        "explanation": explanation.as_list()
    }


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [33]:
for case, result in lime_results.items():
    print(f"Case: {case}")
    print(f"Index: {result['index']}")
    print(f"Actual Label: {result['actual']}")
    print(f"Predicted Label: {result['predicted']}")
    print("Explanation:")
    for feature, weight in result["explanation"]:
        print(f"  {feature}: {weight}")
    print("\n")


Case: TN
Index: 0
Actual Label: 0
Predicted Label: 0
Explanation:
  Sex <= 0.00: -0.3977607818676069
  2.00 < Pclass <= 3.00: -0.11161803171612952
  Fare <= 7.90: -0.10701554528640303
  0.00 < Embarked_S <= 1.00: -0.05712755988315855
  SibSp <= 0.00: 0.03509926987828832


Case: FP
Index: 16
Actual Label: 0
Predicted Label: 1
Explanation:
  Sex <= 0.00: -0.4020448381607569
  Fare > 30.70: 0.15007568761909643
  Pclass <= 2.00: 0.11710446522136675
  Age > 36.00: -0.09050541201867704
  Embarked_S <= 0.00: 0.0543326387637738


Case: FN
Index: 5
Actual Label: 1
Predicted Label: 0
Explanation:
  Sex <= 0.00: -0.38509545196286965
  2.00 < Pclass <= 3.00: -0.10807340230051504
  Fare <= 7.90: -0.10033260793577198
  0.00 < Embarked_S <= 1.00: -0.059081979311250796
  22.00 < Age <= 28.00: -0.03387059554751807


Case: TP
Index: 1
Actual Label: 1
Predicted Label: 1
Explanation:
  0.00 < Sex <= 1.00: 0.39844136757499143
  Age <= 22.00: 0.15345053857193064
  Fare > 30.70: 0.13465997892098738
  Pclass 